# Data Cleaning and Preprocessing
Complete step-by-step pipeline for the `flights_sample_3m.csv` dataset.

## Step 0: Setup and Load Data

In [40]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

dtype_dict = {
    'DOT_CODE': 'int32',
    'FL_NUMBER': 'int32',
    'CRS_DEP_TIME': 'int32',
    'CRS_ARR_TIME': 'int32',
    'CANCELLED': 'float32',
    'DIVERTED': 'float32',
    'DISTANCE': 'float32'
}

print("Loading dataset...")
df = pd.read_csv('flights_sample_3m.csv', dtype=dtype_dict)
print(f"Initial Shape: {df.shape}")
df.head()


Loading dataset...
Initial Shape: (3000000, 32)


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,NaN,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,NaN,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


## Step 1: Remove Rows with Nulls in Critical Columns

In [41]:
# Step 1: Remove rows where critical columns are missing
critical_columns = ['FL_DATE', 'AIRLINE', 'FL_NUMBER', 'ORIGIN', 'DEST', 'DEP_DELAY']

rows_before = len(df)
df = df.dropna(subset=critical_columns)
print(f"Rows removed: {rows_before - len(df)}")
print(f"Shape after Step 1: {df.shape}")


Rows removed: 77644
Shape after Step 1: (2922356, 32)


## Step 2: Convert and Standardize Time Formats

In [42]:
# Step 2: Convert and standardize time formats
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# Extract departure hour (e.g., 1430 -> 14)
df['DEP_HOUR'] = (df['DEP_TIME'] // 100).fillna(0).astype('int32')
df['DEP_TIME'] = df['DEP_TIME'].fillna(0).astype('int32')

print("Time formats standardized.")
df[['FL_DATE', 'DEP_TIME', 'DEP_HOUR']].head()


Time formats standardized.


,FL_DATE,DEP_TIME,DEP_HOUR
0,2019-01-09,1151,11
1,2022-11-19,2114,21
2,2022-07-22,1000,10
3,2023-03-06,1608,16
4,2020-02-23,1838,18


## Step 3: Remove Unwanted Characters from Text Columns


In [43]:
# Step 3: Remove unwanted characters from text columns
text_columns = ['AIRLINE', 'ORIGIN_CITY', 'DEST_CITY']
for col in text_columns:
    df[col] = df[col].astype(str).str.replace('"', '').str.strip()

print("Unwanted characters removed from text columns.")


Unwanted characters removed from text columns.


## Step 4: Handle Type Mismatches

In [44]:
# Step 4: Handle type mismatches (TRY_CAST equivalent)
numeric_columns = ['DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON',
                   'TAXI_IN', 'ARR_TIME', 'ARR_DELAY', 'ELAPSED_TIME', 'AIR_TIME']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Type mismatches handled successfully.")


Type mismatches handled successfully.


## Step 5: Remove Duplicate Records (Composite Key)

In [45]:
# Step 5: Remove duplicate records using composite key
composite_key = ['FL_DATE', 'AIRLINE', 'FL_NUMBER', 'ORIGIN', 'DEST']

rows_before = len(df)
df = df.drop_duplicates(subset=composite_key, keep='first')
print(f"Removed {rows_before - len(df)} duplicate records.")
print(f"Shape after Step 5: {df.shape}")


Removed 0 duplicate records.
Shape after Step 5: (2922356, 33)


## Step 6: Handle Remaining Missing Values (Imputation)
Strategy: Fill with 0.0, median, or group-wise median based on business logic.

In [46]:
# Step 6a: Inspect remaining missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))


                         Missing Count  Missing %
CANCELLATION_CODE              2920860      99.95
DELAY_DUE_CARRIER              2388493      81.73
DELAY_DUE_WEATHER              2388493      81.73
DELAY_DUE_NAS                  2388493      81.73
DELAY_DUE_SECURITY             2388493      81.73
DELAY_DUE_LATE_AIRCRAFT        2388493      81.73
ARR_DELAY                         8554       0.29
ELAPSED_TIME                      8554       0.29
AIR_TIME                          8554       0.29
WHEELS_ON                         2300       0.08
TAXI_IN                           2300       0.08
ARR_TIME                          2298       0.08
TAXI_OUT                          1162       0.04
WHEELS_OFF                        1162       0.04


In [47]:
# Step 6b: Impute missing values with business logic

# DELAY_DUE_* columns: NaN means no delay contribution -> fill with 0.0
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
              'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
df[delay_cols] = df[delay_cols].fillna(0.0)

# CANCELLATION_CODE: NaN means not cancelled -> fill with 'N'
df['CANCELLATION_CODE'] = df['CANCELLATION_CODE'].fillna('N')

# ARR_DELAY: impute with median by airline (group-wise, avoids global bias)
df['ARR_DELAY'] = df.groupby('AIRLINE')['ARR_DELAY'].transform(
    lambda x: x.fillna(x.median())
)

# Remaining timing columns: fill with column median
cols_to_median = ['TAXI_OUT', 'TAXI_IN', 'AIR_TIME', 'ELAPSED_TIME',
                  'WHEELS_OFF', 'WHEELS_ON', 'ARR_TIME', 'CRS_ELAPSED_TIME']
for col in cols_to_median:
    df[col] = df[col].fillna(df[col].median())

remaining = df.isnull().sum().sum()
print(f"Total remaining missing values: {remaining}")
print(f"Shape: {df.shape}")


Total remaining missing values: 0
Shape: (2922356, 33)


## Step 7: Standardize Inconsistent Categories

In [48]:
# Step 7: Standardize inconsistent categories

# Title-case airline names
df['AIRLINE'] = df['AIRLINE'].str.strip().str.title()

# Uppercase airport codes
df['ORIGIN'] = df['ORIGIN'].str.strip().str.upper()
df['DEST'] = df['DEST'].str.strip().str.upper()

# Title-case city names
df['ORIGIN_CITY'] = df['ORIGIN_CITY'].str.strip().str.title()
df['DEST_CITY'] = df['DEST_CITY'].str.strip().str.title()

# Map cancellation codes to readable labels
cancel_map = {
    'A': 'Carrier', 'B': 'Weather', 'C': 'NAS',
    'D': 'Security', 'N': 'Not Cancelled'
}
df['CANCELLATION_CODE'] = df['CANCELLATION_CODE'].map(cancel_map).fillna('Unknown')

print("Unique Airlines:", df['AIRLINE'].nunique())
print("Unique Origins:", df['ORIGIN'].nunique())
print("Cancellation Codes:", df['CANCELLATION_CODE'].unique())


Unique Airlines: 18
Unique Origins: 380
Cancellation Codes: ['Not Cancelled' 'Weather' 'Carrier' 'NAS' 'Security']


## Step 8: Outlier Treatment (Winsorization)
Capping extreme values at 1st-99th percentile. We cap rather than drop because extreme delays are real events.

In [49]:
# Step 8: Outlier treatment using Winsorization (1st-99th percentile capping)
# We cap rather than drop because extreme delays are real events, not errors.

def cap_outliers(df, col, lower_pct=0.01, upper_pct=0.99):
    lower = df[col].quantile(lower_pct)
    upper = df[col].quantile(upper_pct)
    original_range = f"[{df[col].min():.1f}, {df[col].max():.1f}]"
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: {original_range} -> capped to [{lower:.1f}, {upper:.1f}]")
    return df

outlier_cols = ['DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME', 'ELAPSED_TIME']

print("Outlier Capping (Winsorization at 1st-99th Percentile):")
for col in outlier_cols:
    df = cap_outliers(df, col)

print(f"Shape after outlier treatment: {df.shape}")


Outlier Capping (Winsorization at 1st-99th Percentile):
DEP_DELAY: [-90.0, 2966.0] -> capped to [-14.0, 191.0]
ARR_DELAY: [-96.0, 2934.0] -> capped to [-37.0, 189.0]
TAXI_OUT: [1.0, 184.0] -> capped to [6.0, 52.0]
TAXI_IN: [1.0, 249.0] -> capped to [2.0, 33.0]
AIR_TIME: [8.0, 692.0] -> capped to [23.0, 334.0]
ELAPSED_TIME: [15.0, 739.0] -> capped to [42.0, 365.0]
Shape after outlier treatment: (2922356, 33)


## Step 9: Datetime Feature Engineering
Extract Year, Month, Day, DayOfWeek, Quarter and time-of-day bins from FL_DATE.

In [50]:
# Step 9: Full datetime feature engineering

df['YEAR']        = df['FL_DATE'].dt.year
df['MONTH']       = df['FL_DATE'].dt.month
df['DAY']         = df['FL_DATE'].dt.day
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek   # 0=Monday, 6=Sunday
df['IS_WEEKEND']  = df['DAY_OF_WEEK'].isin([5, 6]).astype(int)
df['QUARTER']     = df['FL_DATE'].dt.quarter

# Bin departure hour into time-of-day buckets
def get_time_of_day(hour):
    if hour < 6:    return 'Red-Eye'
    elif hour < 12: return 'Morning'
    elif hour < 17: return 'Afternoon'
    elif hour < 21: return 'Evening'
    else:           return 'Night'

df['DEP_TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

print("Datetime features created.")
print(df[['FL_DATE', 'YEAR', 'MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'QUARTER', 'DEP_TIME_OF_DAY']].head())


Datetime features created.
     FL_DATE  YEAR  MONTH  DAY_OF_WEEK  IS_WEEKEND  QUARTER DEP_TIME_OF_DAY
0 2019-01-09  2019      1            2           0        1         Morning
1 2022-11-19  2022     11            5           1        4           Night
2 2022-07-22  2022      7            4           0        3         Morning
3 2023-03-06  2023      3            0           0        1       Afternoon
4 2020-02-23  2020      2            6           1        1         Evening


## Step 10: Derived Features and KPIs
Create business-meaningful columns: IS_DELAYED, DELAY_SEVERITY, ROUTE, EFFICIENCY_RATIO, IS_PEAK_SEASON.

In [51]:
# Step 10: Create derived features and KPIs

# KPI 1: IS_DELAYED - FAA standard (>15 min arrival delay)
df['IS_DELAYED'] = (df['ARR_DELAY'] > 15).astype(int)

# KPI 2: DELAY_SEVERITY - Categorical classification of delay
def classify_delay(delay):
    if delay <= 0:    return 'On Time / Early'
    elif delay <= 15: return 'Minor Delay'
    elif delay <= 60: return 'Moderate Delay'
    else:             return 'Severe Delay'

df['DELAY_SEVERITY'] = df['ARR_DELAY'].apply(classify_delay)

# KPI 3: ROUTE - Combined origin-destination for route-level analysis
df['ROUTE'] = df['ORIGIN'] + '-' + df['DEST']

# KPI 4: TOTAL_GROUND_TIME - Taxi out + Taxi in (non-flying time)
df['TOTAL_GROUND_TIME'] = df['TAXI_OUT'] + df['TAXI_IN']

# KPI 5: EFFICIENCY_RATIO - Actual vs Scheduled elapsed time (>1 = slower than planned)
df['EFFICIENCY_RATIO'] = (df['ELAPSED_TIME'] / df['CRS_ELAPSED_TIME']).round(3)

# KPI 6: IS_PEAK_SEASON - Summer (Jun-Aug) and Holiday (Nov-Dec)
df['IS_PEAK_SEASON'] = df['MONTH'].isin([6, 7, 8, 11, 12]).astype(int)

print("Derived Features and KPIs created:")
print(df[['AIRLINE', 'ROUTE', 'ARR_DELAY', 'IS_DELAYED', 'DELAY_SEVERITY',
          'TOTAL_GROUND_TIME', 'EFFICIENCY_RATIO', 'IS_PEAK_SEASON']].head(8))


Derived Features and KPIs created:
                  AIRLINE    ROUTE  ARR_DELAY  IS_DELAYED   DELAY_SEVERITY  \
0   United Air Lines Inc.  FLL-EWR      -14.0           0  On Time / Early   
1    Delta Air Lines Inc.  MSP-SEA       -5.0           0  On Time / Early   
2   United Air Lines Inc.  DEN-MSP        0.0           0  On Time / Early   
3    Delta Air Lines Inc.  MSP-SFO       24.0           1   Moderate Delay   
4        Spirit Air Lines  MCO-DFW       -1.0           0  On Time / Early   
5  Southwest Airlines Co.  DAL-OKC      141.0           1     Severe Delay   
6  American Airlines Inc.  DCA-BOS      -29.0           0  On Time / Early   
7        Republic Airline  HSV-DCA       23.0           1   Moderate Delay   

   TOTAL_GROUND_TIME  EFFICIENCY_RATIO  IS_PEAK_SEASON  
0               23.0             0.946               0  
1               42.0             1.004               1  
2               25.0             0.949               1  
3               36.0             1

## Step 11: Categorical Encoding
Label-encode high-cardinality text columns for ML readiness. Original text columns are preserved for EDA.

In [52]:
# Step 11: Categorical encoding using Label Encoding (for ML readiness)
# We keep the original text columns for EDA and add encoded versions

label_encode_cols = ['AIRLINE', 'ORIGIN', 'DEST', 'DEP_TIME_OF_DAY',
                     'DELAY_SEVERITY', 'CANCELLATION_CODE']

le = LabelEncoder()
for col in label_encode_cols:
    encoded_col = col + '_ENC'
    df[encoded_col] = le.fit_transform(df[col].astype(str))
    print(f"{col} -> {encoded_col} | Sample classes: {list(le.classes_)[:4]}...")

print("Encoding complete.")


AIRLINE -> AIRLINE_ENC | Sample classes: ['Alaska Airlines Inc.', 'Allegiant Air', 'American Airlines Inc.', 'Delta Air Lines Inc.']...
ORIGIN -> ORIGIN_ENC | Sample classes: ['ABE', 'ABI', 'ABQ', 'ABR']...
DEST -> DEST_ENC | Sample classes: ['ABE', 'ABI', 'ABQ', 'ABR']...
DEP_TIME_OF_DAY -> DEP_TIME_OF_DAY_ENC | Sample classes: ['Afternoon', 'Evening', 'Morning', 'Night']...
DELAY_SEVERITY -> DELAY_SEVERITY_ENC | Sample classes: ['Minor Delay', 'Moderate Delay', 'On Time / Early', 'Severe Delay']...
CANCELLATION_CODE -> CANCELLATION_CODE_ENC | Sample classes: ['Carrier', 'NAS', 'Not Cancelled', 'Security']...
Encoding complete.


## Step 12: Numerical Scaling (StandardScaler)
Normalize continuous features to mean=0, std=1. Required for distance-based models (KNN, SVM, Logistic Regression).

In [53]:
# # Step 12: Standard scaling for continuous numerical features
# # Mean=0, Std=1 - required for KNN, SVM, Logistic Regression etc.

# scale_cols = ['DEP_DELAY', 'ARR_DELAY', 'DISTANCE', 'AIR_TIME',
#               'TAXI_OUT', 'TAXI_IN', 'TOTAL_GROUND_TIME', 'EFFICIENCY_RATIO']

# scaler = StandardScaler()
# scaled_values = scaler.fit_transform(df[scale_cols])

# scaled_df = pd.DataFrame(
#     scaled_values,
#     columns=[c + '_SCALED' for c in scale_cols],
#     index=df.index
# )
# df = pd.concat([df, scaled_df], axis=1)

# print("Scaled columns added:")
# print(df[[c + '_SCALED' for c in scale_cols]].describe().round(3))


## Step 13: Drop Redundant / Low-Value Columns

In [54]:
# Step 13: Drop redundant / low-value columns
cols_to_drop = ['AIRLINE_DOT', 'DOT_CODE']

# Also drop any column still missing >80% of values
missing_pct = df.isnull().mean() * 100
high_missing = missing_pct[missing_pct > 80].index.tolist()
cols_to_drop.extend(high_missing)
cols_to_drop = list(set(cols_to_drop))

df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Dropped: {cols_to_drop}")
print(f"Final Dataset Shape: {df.shape}")
print(f"Total Columns: {len(df.columns)}")
print(list(df.columns))


Dropped: ['DOT_CODE', 'AIRLINE_DOT']
Final Dataset Shape: (2922356, 50)
Total Columns: 50
['FL_DATE', 'AIRLINE', 'AIRLINE_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'DEP_HOUR', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'IS_WEEKEND', 'QUARTER', 'DEP_TIME_OF_DAY', 'IS_DELAYED', 'DELAY_SEVERITY', 'ROUTE', 'TOTAL_GROUND_TIME', 'EFFICIENCY_RATIO', 'IS_PEAK_SEASON', 'AIRLINE_ENC', 'ORIGIN_ENC', 'DEST_ENC', 'DEP_TIME_OF_DAY_ENC', 'DELAY_SEVERITY_ENC', 'CANCELLATION_CODE_ENC']


## Step 14: Save the Cleaned Dataset

In [55]:
# Step 14: Save the final cleaned dataset

# CSV for broad compatibility
df.to_csv('flights_cleaned.csv', index=False)
print("Saved: flights_cleaned.csv")

# Parquet for fast reloading (smaller size, preserves dtypes)
df.to_parquet('flights_cleaned.parquet', index=False)
print("Saved: flights_cleaned.parquet")

print(f"Final clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")


Saved: flights_cleaned.csv
Saved: flights_cleaned.parquet
Final clean dataset: 2,922,356 rows x 50 columns
